# Data Summary + Analysis

## Set Up

Import libraries/packages + cleaned data

In [ ]:
# Libraries/packages
import sys

sys.path.append("../")
from src.data_utils import get_feature_lists
from src.config import BASE_PATH
from src.summary_analysis import generate_summary_table
import warnings
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import pandas as pd

Data

In [ ]:
## HOPKINS data
cleaned_data = pd.read_parquet(
    BASE_PATH / "data/raw/hopkins_ORN_extra_clean.parquet"
).drop("ID", axis=1)
X = cleaned_data.drop("ORN", axis=1)
y = cleaned_data["ORN"]

Re-order a bit

In [ ]:
reordered_cols = [
    ##Pre-Op
    "AGE",
    "BMI",
    "SEX",
    "Diabetes",
    "ASA",
    "PRIOREX",
    "PRECT",
    ## Disease
    "RECUR",
    "SITE",
    "SIZE",
    "LYMPH",
    "STAGE",
    "DEFECT",
    "SECONDPRIMARY",
    ## Surg
    "LENGTH",
    "JEWER",
    "OSTEOTOMY",
    "PLATE",
    "FLAP",
    "TRANSFUS",
    "ISCHEMICTIME",
    "OPTIME",
    ## Immediate post-op
    "REOP",
    "LOHS",
    "POSTCT",
    "WOUNDINF",
    "HGB",
    "ALB",
    ## Long term post-op
    "EXPOSURE",
    "MEDUSED",
    "SURGUSED",
    "PLATETIME",
    "FOLLOWTIME",
    ## Misc
    "RADTIME",
]

X_ordered = X[reordered_cols].copy()

Classify features by data type

In [ ]:
##Imported func from src
feature_lists = get_feature_lists(X_ordered)
binary_cols = feature_lists["Binary"]
numerical_cols = feature_lists["Numerical"]
nominal_cols = feature_lists["Nominal"]
ordinal_cols = feature_lists["Ordinal"]

Impute

In [ ]:
imputer = IterativeImputer(
    estimator=None,  # default = BayesianRidge
    initial_strategy="median",
    max_iter=10,
    sample_posterior=False,  # deterministic
)
X_imp = X_ordered.copy()
imputed_values = imputer.fit_transform(X_imp[numerical_cols])
X_imp[numerical_cols] = imputed_values
## Ensure no NAs
assert X_imp.isna().sum().sum() == 0

## Summary

In [ ]:
# Create all_categories specific to THIS dataset
all_categories = {}
for col in nominal_cols + binary_cols + ordinal_cols:
    all_categories[col] = X_imp[col].unique()
## Get summary
summary_df = generate_summary_table(
    X_df_final=X_imp,
    X_df_og=X_ordered,
    outcome_data=y,
    data_type="hopkins_test",
    all_categories=all_categories,
    feature_dict=feature_lists,
)

Export

In [ ]:
## Set up path
export_path = BASE_PATH / "results/tables/summary.xlsx"
if export_path.exists():
    export_path.unlink()
    warnings.warn(f"Over-writing folder at path {export_path}")
export_path.parent.mkdir(exist_ok=True, parents=True)
## Export
summary_df.to_excel(export_path)